In [ ]:
import json
import xml.etree.ElementTree as ET
import requests

BASE, NS = "https://api.ibb.gov.tr/iett", "http://tempuri.org/"

def _soap(path, method, params):
    body = "".join(f"<{k}>{v}</{k}>" for k, v in params.items())
    env = ('<?xml version="1.0" encoding="utf-8"?>'
           '<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">'
           f'<soap:Body><{method} xmlns="{NS}">{body}</{method}></soap:Body></soap:Envelope>')
    r = requests.post(f"{BASE}/{path}", data=env.encode("utf-8"),
                      headers={"Content-Type": "text/xml; charset=utf-8",
                               "SOAPAction": f"{NS}{method}"}, timeout=30)
    r.raise_for_status()
    return ET.fromstring(r.content)

In [ ]:
import pandas as pd

sefer = pd.DataFrame(veri)
print(sefer.shape)
sefer.head(20)

In [ ]:
sefer["saat"] = sefer["DT"].str[:2].astype(int)

print("gün tipleri:", sefer["SGUNTIPI"].value_counts().to_dict())
print("yönler:", sefer["SYON"].value_counts().to_dict())
print("servis tipi:", sefer["SSERVISTIPI"].value_counts().to_dict())

arz = sefer.groupby(["SHATKODU", "SGUNTIPI", "saat"]).size().reset_index(name="sefer_sayisi")
arz

In [ ]:
root = _soap("UlasimAnaVeri/HatDurakGuzergah.asmx", "GetHat_json", {"HatKodu": ""})
res = next(el for el in root.iter() if el.tag.endswith("Result"))
hatlar = pd.DataFrame(json.loads(res.text))

print("boyut:", hatlar.shape)
hatlar.head(10)

In [ ]:
def yolculuk_hat(tarih):
    root = _soap("ibb/ibb360.asmx", "GetIettYolculukHat_json", {"Tarih": tarih})
    res = next(el for el in root.iter() if el.tag.endswith("Result"))
    return pd.DataFrame(json.loads(res.text))

yeni = None
for t in ["2026-08-04", "202608-04", "20260804"]:
    try:
        yeni = yolculuk_hat(t)
        print(t, "→ ÇALIŞTI, boyut:", yeni.shape)
        break
    except Exception as e:
        print(t, "→", type(e).__name__, str(e)[:60])

yeni.head(10) if yeni is not None else "hiçbiri çalışmadı"

In [ ]:
talep = pd.read_csv("cikti/talep_ozeti.csv")

eski = (talep.groupby("line_name")["number_of_passenger"].sum()
        .sort_values(ascending=False).head(50).reset_index())
eski["eski_sira"] = range(1, 51)
eski = eski.rename(columns={"line_name": "hat", "number_of_passenger": "ekim2024"})

bugun = yeni.dropna(subset=["Hat"]).reset_index(drop=True)
bugun["yeni_sira"] = range(1, len(bugun) + 1)
bugun = bugun.rename(columns={"Hat": "hat", "Yolculuk": "bugun"})[["hat", "bugun", "yeni_sira"]]

kars = eski.merge(bugun, on="hat", how="outer")
kars["sira_farki"] = kars["yeni_sira"] - kars["eski_sira"]

kars.sort_values("eski_sira").head(30)

In [ ]:
for t in ["2025-10-07", "2024-10-15", "2023-10-10"]:
    try:
        d = yolculuk_hat(t)
        print(t, "→ ÇALIŞTI", d.shape, "| ilk 3:", d["Hat"].head(3).tolist())
    except Exception as e:
        print(t, "→", type(e).__name__, str(e)[:50])

In [ ]:
e2025 = yolculuk_hat("2025-10-07").dropna(subset=["Hat"]).reset_index(drop=True)
e2025["sira_2025"] = range(1, len(e2025) + 1)
e2025 = e2025.rename(columns={"Hat": "hat", "Yolculuk": "yolculuk_2025"})[["hat", "yolculuk_2025", "sira_2025"]]

kars2 = eski.merge(e2025, on="hat", how="outer")
kars2["sira_farki"] = kars2["sira_2025"] - kars2["eski_sira"]

ortak = kars2.dropna(subset=["eski_sira", "sira_2025"])
print("iki listede de olan hat:", len(ortak), "/ 50")
print("sıra korelasyonu:", round(ortak["eski_sira"].corr(ortak["sira_2025"], method="spearman"), 3))
print("ortalama sıra kayması:", round(ortak["sira_farki"].abs().mean(), 1))

kars2.sort_values("eski_sira").head(30)

In [ ]:
%pip install scipy

In [ ]:
import time

def sefer_saatleri(hat_kodu):
    root = _soap("UlasimAnaVeri/PlanlananSeferSaati.asmx", "GetPlanlananSeferSaati_json", {"HatKodu": hat_kodu})
    res = next(el for el in root.iter() if el.tag.endswith("Result"))
    return json.loads(res.text)

parcalar = []
hatasiz, hatali = 0, 0

for i, kod in enumerate(hatlar["SHATKODU"], 1):
    try:
        kayitlar = sefer_saatleri(kod)
        if kayitlar:
            parcalar.append(pd.DataFrame(kayitlar))
        hatasiz += 1
    except Exception:
        hatali += 1
    if i % 50 == 0:
        print(f"{i}/{len(hatlar)} — hatasız: {hatasiz}, hatalı: {hatali}")
    time.sleep(0.15)

sefer_ham = pd.concat(parcalar, ignore_index=True)
print("toplam kalkış:", len(sefer_ham))
sefer_ham.head()

In [ ]:
sefer_ham.to_csv("cikti/sefer_ham_20260806.csv", index=False)
print("kaydedildi:", len(sefer_ham), "satır")

In [ ]:
sefer_ham["saat"] = pd.to_numeric(sefer_ham["DT"].str[:2], errors="coerce")

print("gün tipleri:", sefer_ham["SGUNTIPI"].value_counts().to_dict())
print("servis tipleri:", sefer_ham["SSERVISTIPI"].value_counts().to_dict())
print("saat aralığı:", sefer_ham["saat"].min(), "-", sefer_ham["saat"].max())
print("saati okunamayan:", sefer_ham["saat"].isna().sum())
print("farklı hat:", sefer_ham["SHATKODU"].nunique())

sefer_ham.sample(10)

In [ ]:
gun_map_arz = {"I": "WEEKDAYS", "C": "SATURDAY", "P": "SUNDAY"}
sefer_ham["gun_tipi"] = sefer_ham["SGUNTIPI"].map(gun_map_arz)

arz = (sefer_ham
       .groupby(["SHATKODU", "gun_tipi", "saat"])
       .size()
       .reset_index(name="sefer_sayisi")
       .rename(columns={"SHATKODU": "hat"}))

arz.to_csv("cikti/arz_ozeti.csv", index=False)

print("arz tablosu:", arz.shape)
print("gün tipi dağılımı:", arz["gun_tipi"].value_counts().to_dict())
arz.head(20)

In [ ]:
talep = pd.read_csv("cikti/talep_ozeti.csv")
arz = pd.read_csv("cikti/arz_ozeti.csv")

talep = talep.rename(columns={"line_name": "hat",
                              "transition_hour": "saat",
                              "number_of_passenger": "yolcu_ay"})

talep["hat"] = talep["hat"].astype(str)
arz["hat"] = arz["hat"].astype(str)

gun_sayisi = {"WEEKDAYS": 23, "SATURDAY": 4, "SUNDAY": 4}
talep["yolcu_gunluk"] = talep["yolcu_ay"] / talep["gun_tipi"].map(gun_sayisi)

df = talep.merge(arz, on=["hat", "gun_tipi", "saat"], how="inner")
df["doluluk"] = (df["yolcu_gunluk"] / df["sefer_sayisi"]).round(1)

print("talep satır:", len(talep), "| arz satır:", len(arz), "| eşleşen:", len(df))
print("eşleşen hat sayısı:", df["hat"].nunique())

df.sort_values("doluluk", ascending=False).head(20)

In [ ]:
hat_ozet = (df[df["gun_tipi"] == "WEEKDAYS"]
            .groupby("hat")[["yolcu_gunluk", "sefer_sayisi"]].sum())
hat_ozet["ort_doluluk"] = (hat_ozet["yolcu_gunluk"] / hat_ozet["sefer_sayisi"]).round(1)
hat_ozet = hat_ozet.sort_values("ort_doluluk", ascending=False)

print("toplam hat:", len(hat_ozet))
print("110 üstü:", (hat_ozet["ort_doluluk"] > 110).sum())
print("200 üstü:", (hat_ozet["ort_doluluk"] > 200).sum())
print("500 üstü:", (hat_ozet["ort_doluluk"] > 500).sum())
print()
print(hat_ozet["ort_doluluk"].describe().round(1).to_dict())

hat_ozet.head(40)

In [ ]:
df_temiz = df[~df["hat"].isin(["34", "34A"])].copy()

print("saatlik doluluk dağılımı:")
print(df_temiz["doluluk"].describe().round(1).to_dict())
print()
print("110 üstü satır:", (df_temiz["doluluk"] > 110).sum(), "/", len(df_temiz))
print("110 üstü farklı hat:", df_temiz[df_temiz["doluluk"] > 110]["hat"].nunique())
print("20 altı satır:", (df_temiz["doluluk"] < 20).sum())

df_temiz.sort_values("doluluk", ascending=False).head(30)


In [ ]:
GECERLI_GUNLER = ["2024-10-02", "2024-10-04", "2024-10-16", "2024-10-17", "2024-10-18"]

parcalar = []
for parca in pd.read_csv("veri/yolculuk/hourly_transportation_202410.csv", chunksize=500000):
    o = parca[(parca["road_type"] == "OTOYOL") &
              (parca["transition_date"].isin(GECERLI_GUNLER))]
    if len(o):
        parcalar.append(o.groupby(["line_name", "transition_hour"])["number_of_passenger"].sum().reset_index())

gercek = pd.concat(parcalar).groupby(["line_name", "transition_hour"])["number_of_passenger"].sum().reset_index()
gercek["yolcu_gunluk"] = (gercek["number_of_passenger"] / len(GECERLI_GUNLER)).round(1)
gercek = gercek.rename(columns={"line_name": "hat", "transition_hour": "saat"})[["hat", "saat", "yolcu_gunluk"]]

gercek.to_csv("cikti/gercek_haftaici_profil.csv", index=False)

print("hat sayısı:", gercek["hat"].nunique(), "| satır:", len(gercek))
print("toplam günlük yolcu:", round(gercek["yolcu_gunluk"].sum()))
gercek.sort_values("yolcu_gunluk", ascending=False).head(15)

In [ ]:
import numpy as np

CARPAN = {"SATURDAY": 0.72, "SUNDAY": 0.52}

arz = pd.read_csv("cikti/arz_ozeti.csv")
arz["hat"] = arz["hat"].astype(str)
gercek["hat"] = gercek["hat"].astype(str)

hat_toplam = gercek.groupby("hat")["yolcu_gunluk"].sum()
rng = np.random.default_rng(42)
parcalar = []

for gt, carpan in CARPAN.items():
    a = arz[arz["gun_tipi"] == gt].copy()
    a["pay"] = a.groupby("hat")["sefer_sayisi"].transform(lambda s: s / s.sum())
    a["hat_toplam"] = a["hat"].map(hat_toplam)
    a = a.dropna(subset=["hat_toplam"])
    a["yolcu_gunluk"] = (a["hat_toplam"] * carpan * a["pay"] *
                         rng.normal(1.0, 0.07, len(a))).clip(lower=0).round(1)
    parcalar.append(a[["hat", "gun_tipi", "saat", "yolcu_gunluk"]])

sentetik = pd.concat(parcalar, ignore_index=True)
sentetik["kaynak"] = "sentetik"

print("üretilen satır:", len(sentetik))
print("günlük toplam:", sentetik.groupby("gun_tipi")["yolcu_gunluk"].sum().round(0).to_dict())
sentetik.head(15)

In [ ]:
gercek_tam = gercek.copy()
gercek_tam["gun_tipi"] = "WEEKDAYS"
gercek_tam["kaynak"] = "gercek"
gercek_tam = gercek_tam[["hat", "gun_tipi", "saat", "yolcu_gunluk", "kaynak"]]

talep_final = pd.concat([gercek_tam, sentetik], ignore_index=True)
talep_final.to_csv("cikti/talep_final.csv", index=False)

print("toplam satır:", len(talep_final))
print(talep_final.groupby(["gun_tipi", "kaynak"])["yolcu_gunluk"].agg(["count", "sum"]).round(0))

In [ ]:
son = talep_final.merge(arz, on=["hat", "gun_tipi", "saat"], how="inner")
son["doluluk"] = (son["yolcu_gunluk"] / son["sefer_sayisi"]).round(1)
son = son[~son["hat"].isin(["34", "34A"])].copy()
son.to_csv("cikti/doluluk.csv", index=False)

print("satır:", len(son), "| hat:", son["hat"].nunique())
print("doluluk medyanı:", son["doluluk"].median())
print("110 üstü:", (son["doluluk"] > 110).sum())
print("20 altı:", (son["doluluk"] < 20).sum())

son.sort_values("doluluk", ascending=False).head(25)




In [ ]:
oneri = son[son["sefer_sayisi"] >= 3].copy()

print("filtreli satır:", len(oneri), "/", len(son))
print("kaynak bazında medyan:", oneri.groupby("kaynak")["doluluk"].median().round(1).to_dict())

haftaici = oneri[oneri["gun_tipi"] == "WEEKDAYS"].sort_values("doluluk", ascending=False)

print("\nhafta içi (gerçek veri) 110 üstü:", (haftaici["doluluk"] > 110).sum())
print("etkilenen hat:", haftaici[haftaici["doluluk"] > 110]["hat"].nunique())

haftaici.head(25)


In [ ]:
oneri_artir = haftaici[haftaici["doluluk"] > 110].copy()
oneri_artir["gereken_sefer"] = np.ceil(oneri_artir["yolcu_gunluk"] / 110).astype(int)
oneri_artir["ek_sefer"] = oneri_artir["gereken_sefer"] - oneri_artir["sefer_sayisi"]
oneri_artir = oneri_artir[["hat", "saat", "yolcu_gunluk", "sefer_sayisi", "gereken_sefer", "ek_sefer", "doluluk"]]
oneri_artir.to_csv("cikti/oneri_sefer_artir.csv", index=False)

print("öneri sayısı:", len(oneri_artir))
print("toplam ek sefer ihtiyacı:", oneri_artir["ek_sefer"].sum())
oneri_artir.head(25)

In [ ]:
import matplotlib.pyplot as plt

HAT = "11ÜS"

d = son[(son["hat"] == HAT) & (son["gun_tipi"] == "WEEKDAYS")].sort_values("saat")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(d["saat"], d["doluluk"], color="#2a78d6", width=0.7)
ax.axhline(110, color="#d03b3b", linestyle="--", linewidth=2)
ax.text(23.4, 110, "kapasite 110", color="#d03b3b", va="center", fontsize=9)

ax.set_title(f"{HAT} — hafta içi, sefer başına düşen yolcu", fontsize=13, pad=12)
ax.set_xlabel("saat")
ax.set_ylabel("kişi / sefer")
ax.set_xticks(range(0, 24))
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
RENK = {"WEEKDAYS": "#2a78d6", "SATURDAY": "#eb6834", "SUNDAY": "#1baf7a"}
ETIKET = {"WEEKDAYS": "Hafta içi (gerçek)", "SATURDAY": "Cumartesi (sentetik)", "SUNDAY": "Pazar (sentetik)"}

fig, ax = plt.subplots(figsize=(10, 4.5))

for gt in ["WEEKDAYS", "SATURDAY", "SUNDAY"]:
    g = son[(son["hat"] == HAT) & (son["gun_tipi"] == gt)].sort_values("saat")
    ax.plot(g["saat"], g["doluluk"], color=RENK[gt], linewidth=2, marker="o",
            markersize=4, label=ETIKET[gt])

ax.axhline(110, color="#d03b3b", linestyle="--", linewidth=2)
ax.text(23.4, 110, "kapasite 110", color="#d03b3b", va="center", fontsize=9)

ax.set_title(f"{HAT} — gün tipine göre sefer başına yolcu", fontsize=13, pad=12)
ax.set_xlabel("saat")
ax.set_ylabel("kişi / sefer")
ax.set_xticks(range(0, 24))
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
SAAT_DUZELTME = {0:1.30, 1:1.30, 2:1.30, 3:1.20, 4:1.00, 5:0.60,
                 6:0.45, 7:0.40, 8:0.45, 9:0.65, 10:1.10, 11:1.20,
                 12:1.25, 13:1.25, 14:1.20, 15:1.15, 16:1.10, 17:1.05,
                 18:1.00, 19:1.05, 20:1.10, 21:1.15, 22:1.25, 23:1.30}

rng = np.random.default_rng(42)
parcalar = []

for gt, carpan in CARPAN.items():
    t = gercek.copy()
    t["ham"] = t["yolcu_gunluk"] * t["saat"].map(SAAT_DUZELTME)
    hafta_toplam = t.groupby("hat")["yolcu_gunluk"].transform("sum")
    ham_toplam = t.groupby("hat")["ham"].transform("sum")
    t["yolcu_gunluk"] = (t["ham"] / ham_toplam * hafta_toplam * carpan *
                         rng.normal(1.0, 0.07, len(t))).clip(lower=0).round(1)
    t["gun_tipi"] = gt
    parcalar.append(t[["hat", "gun_tipi", "saat", "yolcu_gunluk"]])

sentetik = pd.concat(parcalar, ignore_index=True)
sentetik["kaynak"] = "sentetik"

print("üretilen satır:", len(sentetik))
print("günlük toplam:", sentetik.groupby("gun_tipi")["yolcu_gunluk"].sum().round(0).to_dict())
sentetik.head()

In [ ]:
# arz verisinde dönüş (D) seferleri eksik çekilen 52 hat — durak API'si ile doğrulandı
ARZ_EKSIK = ["11D", "11K", "11M", "11ST", "12", "130ŞT", "13H", "145M", "16YK", "19D",
             "22B", "25", "25C", "25S1", "25S2", "29A", "34A", "36", "36Z", "38Z",
             "46KT", "46T", "48S", "48U", "50A", "54HT", "54HŞ", "54P", "54T", "58S",
             "66Z", "70FY", "71T", "73B", "73F", "8Y", "90B", "92K", "9A", "AND1Ş",
             "AND2K", "AND2S", "EL2", "K3", "KM42", "KM43", "KM44", "MK19", "MR42",
             "MR60", "PD1", "UM13"]

gercek_tam = gercek.copy()
gercek_tam["gun_tipi"] = "WEEKDAYS"
gercek_tam["kaynak"] = "gercek"
gercek_tam = gercek_tam[["hat", "gun_tipi", "saat", "yolcu_gunluk", "kaynak"]]

talep_final = pd.concat([gercek_tam, sentetik], ignore_index=True)
talep_final.to_csv("cikti/talep_final.csv", index=False)

son = talep_final.merge(arz, on=["hat", "gun_tipi", "saat"], how="inner")
son["doluluk"] = (son["yolcu_gunluk"] / son["sefer_sayisi"]).round(1)
son = son[~son["hat"].isin(["34", "34A"] + ARZ_EKSIK)].copy()
son.to_csv("cikti/doluluk.csv", index=False)

print("doluluk satır:", len(son), "| medyan:", son["doluluk"].median())

fig, ax = plt.subplots(figsize=(10, 4.5))
for gt in ["WEEKDAYS", "SATURDAY", "SUNDAY"]:
    g = son[(son["hat"] == HAT) & (son["gun_tipi"] == gt)].sort_values("saat")
    ax.plot(g["saat"], g["doluluk"], color=RENK[gt], linewidth=2, marker="o",
            markersize=4, label=ETIKET[gt])
ax.axhline(110, color="#d03b3b", linestyle="--", linewidth=2)
ax.text(23.4, 110, "kapasite 110", color="#d03b3b", va="center", fontsize=9)
ax.set_title(f"{HAT} — gün tipine göre sefer başına yolcu", fontsize=13, pad=12)
ax.set_xlabel("saat")
ax.set_ylabel("kişi / sefer")
ax.set_xticks(range(0, 24))
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
top = (son[(son["gun_tipi"] == "WEEKDAYS") & (son["sefer_sayisi"] >= 3)]
       .nlargest(15, "doluluk").copy())
top["etiket"] = top["hat"] + "  " + top["saat"].astype(str).str.zfill(2) + ":00"

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["etiket"][::-1], top["doluluk"][::-1], color="#d03b3b", height=0.65)
ax.axvline(110, color="#898781", linestyle="--", linewidth=1.5)

for y, v in enumerate(top["doluluk"][::-1]):
    ax.text(v + 6, y, f"{v:.0f}", va="center", fontsize=9, color="#52514e")

ax.set_title("En sıkışık 15 hat + saat — hafta içi, gerçek veri", fontsize=13, pad=12)
ax.set_xlabel("kişi / sefer  (kesikli çizgi: kapasite 110)")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
SUPHELI = ["Gareli", "Ara Dinlen", "Tek Sof S.", "Tek Sof A."]

arz_dar = (sefer_ham[~sefer_ham["SSERVISTIPI"].isin(SUPHELI)]
           .groupby(["SHATKODU", "SGUNTIPI", "saat"]).size()
           .reset_index(name="sefer_sayisi")
           .rename(columns={"SHATKODU": "hat"}))
arz_dar["gun_tipi"] = arz_dar["SGUNTIPI"].map({"I": "WEEKDAYS", "C": "SATURDAY", "P": "SUNDAY"})
arz_dar["hat"] = arz_dar["hat"].astype(str)

son_dar = talep_final.merge(arz_dar[["hat", "gun_tipi", "saat", "sefer_sayisi"]],
                            on=["hat", "gun_tipi", "saat"], how="inner")
son_dar["doluluk"] = (son_dar["yolcu_gunluk"] / son_dar["sefer_sayisi"]).round(1)
son_dar = son_dar[~son_dar["hat"].isin(["34", "34A"])]

print("geniş tanım — satır:", len(son), "medyan:", son["doluluk"].median(),
      "110 üstü:", (son["doluluk"] > 110).sum())
print("dar tanım   — satır:", len(son_dar), "medyan:", son_dar["doluluk"].median(),
      "110 üstü:", (son_dar["doluluk"] > 110).sum())


In [ ]:
for tip in SUPHELI:
    d = sefer_ham[sefer_ham["SSERVISTIPI"] == tip]
    print(f"\n{tip} ({len(d)} kalkış)")
    print(d["saat"].value_counts().sort_index().to_dict())
    

In [ ]:
zirve = (son[son["gun_tipi"] == "WEEKDAYS"]
         .sort_values("doluluk", ascending=False)
         .drop_duplicates("hat")
         .copy())

zirve["aralik_dk"] = (60 / zirve["sefer_sayisi"]).round(1)
supheli = zirve[(zirve["doluluk"] > 200) & (zirve["sefer_sayisi"] >= 3)]

print("toplam hat:", len(zirve))
print("zirve doluluğu 200 üstü:", len(supheli))
supheli[["hat", "saat", "yolcu_gunluk", "sefer_sayisi", "aralik_dk", "doluluk"]].head(20)


In [ ]:
def hat_araclari(hat_no):
    root = _soap("FiloDurum/SeferGerceklesme.asmx", "GetHatOtoKonum_json", {"HatNo": hat_no})
    res = next(el for el in root.iter() if el.tag.endswith("Result"))
    return pd.DataFrame(json.loads(res.text))

a = hat_araclari("11ÜS")
print("11ÜS — şu an yolda:", len(a), "araç")
a.head()

In [ ]:
DENEMELER = [
    {"HatNo": "15B"},
    {"HatKodu": "15B"},
    {"hatNo": "15B"},
    {"HatNo": "15b"},
]

for p in DENEMELER:
    try:
        r = _soap("FiloDurum/SeferGerceklesme.asmx", "GetHatOtoKonum_json", p)
        res = next(el for el in r.iter() if el.tag.endswith("Result"))
        d = pd.DataFrame(json.loads(res.text))
        print(p, "→ ÇALIŞTI", d.shape)
        print(d.head(3))
        break
    except Exception as e:
        print(p, "→", type(e).__name__, str(e)[:70])

In [ ]:
import time

def arac_sayisi(hat_no):
    r = _soap("FiloDurum/SeferGerceklesme.asmx", "GetHatOtoKonum_json", {"HatKodu": hat_no})
    res = next(el for el in r.iter() if el.tag.endswith("Result"))
    return len(json.loads(res.text))

SAAT = 9

kontrol = []
for hat in ["11ÜS", "98", "98KM", "15F", "14B", "500T", "15B", "78"]:
    try:
        n = arac_sayisi(hat)
    except Exception:
        n = None
    s = hatlar.loc[hatlar["SHATKODU"] == hat, "SEFER_SURESI"]
    sure = float(s.iloc[0]) if len(s) else None
    t = arz[(arz["hat"] == hat) & (arz["gun_tipi"] == "WEEKDAYS") & (arz["saat"] == SAAT)]["sefer_sayisi"]
    tarife = int(t.iloc[0]) if len(t) else 0
    tahmin = round(n * 60 / sure, 1) if (n and sure) else None
    kontrol.append({"hat": hat, "yoldaki_arac": n, "sefer_suresi_dk": sure,
                    "tarife_sefer": tarife, "tahmini_sefer": tahmin})
    time.sleep(0.3)

pd.DataFrame(kontrol)


In [ ]:
import json

KAPASITE = 110
GUN_TIPLERI = ["WEEKDAYS", "SATURDAY", "SUNDAY"]

hi = son[(son["gun_tipi"] == "WEEKDAYS") & (son["sefer_sayisi"] >= 3)].copy()
oneri_artir = hi[hi["doluluk"] > KAPASITE].copy()
oneri_artir["gereken_sefer"] = np.ceil(oneri_artir["yolcu_gunluk"] / KAPASITE).astype(int)
oneri_artir["ek_sefer"] = oneri_artir["gereken_sefer"] - oneri_artir["sefer_sayisi"]
oneri_artir.to_csv("cikti/oneri_sefer_artir.csv", index=False)

hat_bilgi = hatlar.set_index("SHATKODU")[["SHATADI", "SEFER_SURESI", "HAT_UZUNLUGU"]].to_dict("index")

def sayi(v, ondalik=1):
    return None if pd.isna(v) else round(float(v), ondalik)

veri_js = {}
for hat, grup in son.groupby("hat"):
    b = hat_bilgi.get(hat, {})
    kayit = {
        "ad": b.get("SHATADI"),
        "sefer_suresi": sayi(b.get("SEFER_SURESI", float("nan"))),
        "uzunluk": sayi(b.get("HAT_UZUNLUGU", float("nan")), 2),
    }
    for gt in GUN_TIPLERI:
        g = grup[grup["gun_tipi"] == gt].set_index("saat").reindex(range(24))
        kayit[gt] = {
            "yolcu": [sayi(v, 0) for v in g["yolcu_gunluk"]],
            "sefer": [sayi(v, 0) for v in g["sefer_sayisi"]],
            "doluluk": [sayi(v) for v in g["doluluk"]],
        }
    veri_js[hat] = kayit

ozet = {
    "olusturma": "2026-08-07",
    "kapasite": KAPASITE,
    "hat_sayisi": len(veri_js),
    "oneri_sayisi": int(len(oneri_artir)),
    "etkilenen_hat": int(oneri_artir["hat"].nunique()),
    "toplam_ek_sefer": int(oneri_artir["ek_sefer"].sum()),
    "talep_kaynagi": "İBB Açık Veri — Ekim 2024, 5 geçerli gün (hafta içi gerçek)",
    "arz_kaynagi": "İETT SOAP API — PlanlananSeferSaati, 6 Ağustos 2026",
    "sentetik_not": "Cumartesi ve pazar verileri sentetiktir (hafta içinin %72 ve %52'si)",
}

with open("cikti/dashboard_veri.json", "w", encoding="utf-8") as f:
    json.dump({"ozet": ozet, "hatlar": veri_js}, f, ensure_ascii=False)

print("yazıldı:", len(veri_js), "hat")
print(ozet)